# Merge data

In this notebook, we will merge our utility, election, and climate data into one file so that we can analyze it and use it for modeling.

In [18]:
import pandas as pd

In [19]:
utility = pd.read_csv("../data/processed/utility_bundled.csv", dtype={"zip": str})
election = pd.read_csv("../data/processed/elections_2024_county.csv", dtype={"county_fips": str})
climate = pd.read_csv("../data/processed/climate_2024.csv", dtype={"county_fips": str})
cross = pd.read_csv("../data/processed/crosswalk.csv", dtype={"ZIP": str, "COUNTY": str})

In [20]:
utility.head()

,zip,utility_name,state,service_type,ownership,res_rate
0,85321,Ajo Improvement Co,AZ,Bundled,Investor Owned,0.125649
1,36560,Alabama Power Co,AL,Bundled,Investor Owned,0.167720
2,36513,Alabama Power Co,AL,Bundled,Investor Owned,0.167720
3,36280,Alabama Power Co,AL,Bundled,Investor Owned,0.167720
4,35473,Alabama Power Co,AL,Bundled,Investor Owned,0.167720


In [21]:
election.head()

,county_fips,state,county_name,DEMOCRAT,REPUBLICAN,totalvotes,rep_share,dem_share,rep_margin
0,01001,ALABAMA,AUTAUGA,7439.0,20484.0,28281,0.724303,0.263039,0.461264
1,01003,ALABAMA,BALDWIN,24934.0,95798.0,122249,0.783630,0.203961,0.579669
2,01005,ALABAMA,BARBOUR,4158.0,5606.0,9855,0.568848,0.421918,0.146930
3,01007,ALABAMA,BIBB,1619.0,7572.0,9257,0.817976,0.174895,0.643081
4,01009,ALABAMA,BLOUNT,2576.0,25354.0,28163,0.900259,0.091468,0.808792


In [22]:
climate.head()

,county_fips,cdd_annual,hdd_annual
0,01001,2513,2003
1,01003,3055,1355
2,01005,2526,1811
3,01007,2294,2212
4,01009,2040,2544


In [23]:
cross.head()

,ZIP,COUNTY,RES_RATIO,BUS_RATIO,OTH_RATIO,TOT_RATIO
0,00501,36103,0.000000,1.00000,0.000000,1.000000
1,01001,25013,1.000000,1.00000,1.000000,1.000000
2,01002,25015,0.992615,0.99886,0.997429,0.993309
3,01002,25011,0.007385,0.00114,0.002571,0.006691
4,01003,25015,1.000000,1.00000,1.000000,1.000000


To merge all of our data, we will need to use the zip-FIPS code crosswalk to map our zip codes in the utility rate data to county FIPS codes, which are used in the climate and election data.

The utility data has multiple rows per zip code, as multiple companies can cover the same area. As a result, there are potentially multiple values of res_rate for one zip code. If we want to map zip codes to county FIPS codes, we will need to compute a single price for each zip code. We will do this by taking the average of prices for each zip.

In [24]:
# compute number of different prices for each zip
unique_price_counts = utility.groupby('zip')['res_rate'].nunique()

# get the distribution of price counts
stats = pd.DataFrame({
    'count': unique_price_counts.value_counts(),
    'percentage': unique_price_counts.value_counts(normalize=True).mul(100).round(2).astype(str) + '%'
}).sort_index()

rates_per_zip = unique_price_counts.mean().item()
print(f"Average unique prices per zip code: {rates_per_zip}")
print("\n=====Distribution of unique prices per zip code=====")
print(stats)

Average unique prices per zip code: 1.5298983507934862

=====Distribution of unique prices per zip code=====
          count percentage
res_rate                  
1         23958     62.13%
2          9960     25.83%
3          3645      9.45%
4           843      2.19%
5           138      0.36%
6            16      0.04%
7             4      0.01%


The majority of zip codes (62.1%) only have 1 rate. For the other zip codes, we will take the mean res_rate for that zip and use that as an estimate for the entire zip code.

Connecticut recently changed from counties to to planning regions. This contains the mapping of the old counties to the new planning regions: https://www.countyhealthrankings.org/health-data/how-to-use-your-county-health-snapshot/changes-to-county-equivalents-for-connecticut

We will use this mapping before merging the datasets. We want to use the legacy FIPS codes because these were used for the election and climate data.

In [25]:
ct_county_to_region = {
    "09001": "09190",  # Fairfield -> Western Connecticut
    "09003": "09110",  # Hartford -> Capitol
    "09005": "09160",  # Litchfield -> Northwest Hills
    "09007": "09130",  # Middlesex -> Lower CT River Valley
    "09009": "09170",  # New Haven -> South Central Connecticut
    "09011": "09180",  # New London -> Southeastern Connecticut
    "09013": "09110",  # Tolland -> Capitol
    "09015": "09150",  # Windham -> Northeastern Connecticut
}

In [26]:
# calculate the mean rate for each zip so each zip has exactly one representative price
utility_zip_avg = utility.groupby('zip')['res_rate'].mean().reset_index()

# map zip codes to counties
utility_mapped = utility_zip_avg.merge(
    cross[['ZIP', 'COUNTY', 'RES_RATIO']], 
    left_on='zip', 
    right_on='ZIP', 
    how='inner'
)

# proportional weighting
utility_mapped['weighted_numerator'] = utility_mapped['res_rate'] * utility_mapped['RES_RATIO']

# aggregate by county
county_stats = utility_mapped.groupby('COUNTY').agg({
    'weighted_numerator': 'sum',
    'RES_RATIO': 'sum'
}).reset_index()

# normalize
county_stats['avg_res_rate'] = county_stats['weighted_numerator'] / county_stats['RES_RATIO']

# final join
county_utility = county_stats[['COUNTY', 'avg_res_rate']].rename(columns={'COUNTY': 'county_fips'})

# recover Connecticut by mapping legacy county FIPS to dominant planning region rates
ct_rate_rows = []
for old_fips, region_fips in ct_county_to_region.items():
    region_row = county_utility[county_utility["county_fips"] == region_fips]
    # keep the old fips so can merge with climate and election data, which uses the old fips codes
    if len(region_row):
        ct_rate_rows.append({
            "county_fips": old_fips,
            "avg_res_rate": region_row["avg_res_rate"].iloc[0],
        })

county_utility = pd.concat(
    [county_utility, pd.DataFrame(ct_rate_rows)],
    ignore_index=True,
)

master_df = election.merge(climate, on='county_fips', how='inner')
master_df = master_df.merge(county_utility, on='county_fips', how='inner')

In [27]:
master_df = master_df.rename(columns={'DEMOCRAT': 'dem_votes', 'REPUBLICAN': 'rep_votes'})

In [28]:
master_df

,county_fips,state,county_name,dem_votes,rep_votes,totalvotes,rep_share,dem_share,rep_margin,cdd_annual,hdd_annual,avg_res_rate
0,01001,ALABAMA,AUTAUGA,7439.0,20484.0,28281,0.724303,0.263039,0.461264,2513,2003,0.167074
1,01003,ALABAMA,BALDWIN,24934.0,95798.0,122249,0.783630,0.203961,0.579669,3055,1355,0.142042
2,01005,ALABAMA,BARBOUR,4158.0,5606.0,9855,0.568848,0.421918,0.146930,2526,1811,0.161961
3,01007,ALABAMA,BIBB,1619.0,7572.0,9257,0.817976,0.174895,0.643081,2294,2212,0.164650
4,01009,ALABAMA,BLOUNT,2576.0,25354.0,28163,0.900259,0.091468,0.808792,2040,2544,0.159236
...,...,...,...,...,...,...,...,...,...,...,...,...
3085,56037,WYOMING,SWEETWATER,3731.0,12541.0,16698,0.751048,0.223440,0.527608,265,7966,0.136690
3086,56039,WYOMING,TETON,8748.0,4134.0,13286,0.311155,0.658437,-0.347283,6,10282,0.088213
3087,56041,WYOMING,UINTA,1561.0,7282.0,9089,0.801188,0.171746,0.629442,117,8514,0.134437
3088,56043,WYOMING,WASHAKIE,656.0,3125.0,3877,0.806036,0.169203,0.636833,500,7017,0.132696


In [29]:
master_df[master_df['state'] == 'CONNECTICUT']

,county_fips,state,county_name,dem_votes,rep_votes,totalvotes,rep_share,dem_share,rep_margin,cdd_annual,hdd_annual,avg_res_rate
279,09001,CONNECTICUT,FAIRFIELD,267019.0,178263.0,452303,0.394123,0.590354,-0.196231,853,4903,0.287646
280,09003,CONNECTICUT,HARTFORD,259366.0,162572.0,429325,0.378669,0.604125,-0.225456,765,5305,0.284101
281,09005,CONNECTICUT,LITCHFIELD,47940.0,56452.0,105969,0.532722,0.452396,0.080325,528,5866,0.285075
282,09007,CONNECTICUT,MIDDLESEX,54173.0,41654.0,97519,0.427137,0.555512,-0.128375,713,5186,0.282682
283,09009,CONNECTICUT,NEW HAVEN,218981.0,171435.0,397987,0.430755,0.550221,-0.119466,747,5185,0.322816
284,09011,CONNECTICUT,NEW LONDON,76190.0,58858.0,137500,0.428058,0.554109,-0.126051,662,5160,0.247599
285,09013,CONNECTICUT,TOLLAND,43311.0,36773.0,81520,0.451092,0.531293,-0.080201,593,5543,0.284101
286,09015,CONNECTICUT,WINDHAM,25073.0,30911.0,56887,0.543375,0.440751,0.102625,577,5526,0.284871


In [30]:
print(f"Rows: {len(master_df):,}")
print(f"States covered: {master_df['state'].nunique()}")
print(f"\nMissing values:")
print(master_df.isna().sum())
print(f"\nKey distributions:")
print(master_df[['avg_res_rate', 'cdd_annual', 'hdd_annual', 'rep_share']].describe())

Rows: 3,090
States covered: 48

Missing values:
county_fips     0
state           0
county_name     0
dem_votes       0
rep_votes       0
totalvotes      0
rep_share       0
dem_share       0
rep_margin      0
cdd_annual      0
hdd_annual      0
avg_res_rate    1
dtype: int64

Key distributions:
       avg_res_rate   cdd_annual    hdd_annual    rep_share
count   3089.000000  3090.000000   3090.000000  3090.000000
mean       0.149336  1479.525890   4250.228803     0.668481
std        0.041533   947.324694   2021.554263     0.156914
min        0.048969     0.000000    129.000000     0.111403
25%        0.127030   740.250000   2631.500000     0.580645
50%        0.142384  1286.000000   4073.500000     0.700872
75%        0.157612  2057.250000   5690.250000     0.790894
max        0.432490  5082.000000  10810.000000     0.959016


In [31]:
master_df[master_df['avg_res_rate'].isna()]

,county_fips,state,county_name,dem_votes,rep_votes,totalvotes,rep_share,dem_share,rep_margin,cdd_annual,hdd_annual,avg_res_rate
2623,48301,TEXAS,LOVING,10.0,86.0,97,0.886598,0.103093,0.783505,3063,1922,NaN


In [32]:
cross[cross['COUNTY'] == '48301']

,ZIP,COUNTY,RES_RATIO,BUS_RATIO,OTH_RATIO,TOT_RATIO
46563,79754,48301,0.0,1.0,0.0,1.0


This row ended up with NaN for avg_res_rate because RES_RATIO = 0, which caused a divide-by-zero during normalization. A RES_RATIO of 0 means HUD doesn't classify any addresses in this ZIP as residential — which makes the county irrelevant for analyzing residential electricity rates anyway.

In [33]:
print(f"Before: {len(master_df):,} rows")
master_df = master_df.dropna(subset=["avg_res_rate"]).reset_index(drop=True)
print(f"After:  {len(master_df):,} rows")

Before: 3,090 rows
After:  3,089 rows


In [34]:
master_df.to_csv("../data/processed/utility-climate-election.csv", index=False)